# Phase 2 — Train Argus-RN34 + Argus-CCT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/05_phase_2_train.ipynb)

Trains the two paired production disease classifiers on the scaled NeuroPainting crop set and pushes per-epoch checkpoints to Hugging Face.

**How to run:** run the shared setup (cells 1-5), then **§ A** on Colab Pro (L4) for **Argus-RN34** (ImageNet-pretrained, converges fast) and **§ B** on a rented A100 for **Argus-CCT** (from scratch, more epochs). Both read the same data loaders and report best-epoch val accuracy from the train log.

**Spec:** `docs/superpowers/specs/2026-05-12-argus-cells-design.md` §2, §5. **Plan:** `docs/superpowers/plans/2026-06-09-argus-cells-phase-2.md` (Group C).

In [ ]:
# Install the package code only (--no-deps) so pip never touches Colab's pre-built
# numpy / scipy / scikit-learn. Upgrading those mid-session breaks the numpy<->scipy
# ABI (the "_blas_supports_fpe" / "_center" ImportErrors).
!pip install -q --upgrade --no-deps git+https://github.com/PatrickJReed/argus-cells.git@main
# Pure-Python deps Colab may lack. pytorch-msssim is needed because this notebook
# imports argus_cells.training (its segmentation_loss uses it); it depends only
# on torch, which Colab has. None of these pull a numpy-linked package.
!pip install -q boto3 huggingface_hub pytorch-msssim

import sys
for _m in list(sys.modules):
    if _m == 'argus_cells' or _m.startswith('argus_cells.'):
        del sys.modules[_m]
print('install + cache purge done')

In [ ]:
import os
from pathlib import Path
import torch

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF login: OK (Colab secret)')
except Exception as e:
    print(f'HF login skipped ({type(e).__name__}); HF push will be disabled')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_BASE = Path('/content/drive/MyDrive/cerberus-neuro/cache/v0')
    CACHE_DIR = Path('/content/cerberus-cache')  # fast local SSD for the ~90 GB TIFF cache
except Exception:
    CKPT_BASE = Path.home() / '.cache' / 'cerberus-neuro' / 'v0'
    CACHE_DIR = CKPT_BASE

CKPT_BASE.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'CACHE_DIR: {CACHE_DIR}')
print(f'CKPT_BASE: {CKPT_BASE}')
print(f'device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})')

In [ ]:
from argus_cells.data import (
    BUCKET, WORKSPACE_PREFIX, CHANNEL_ORDER,
    build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset,
)

# Scale knobs. Crops scale via CROPS_PER_SITE (storage-neutral: a 256-crop is a
# tile of an already-downloaded 2160x2160 site), NOT via more sites (each site is
# ~56 MB of TIFF). 48 wells x 9 sites = ~1728 sites (~90 GB cache, same as Phase 1
# so the prefetch reuses that cache); CROPS_PER_SITE=30 lifts the crop count to
# ~40-50k (3x Phase 1's ~16k), inside the spec's 50-100k target band. Reduce to
# CROPS_PER_SITE=10 for a fast ~16k-crop validation pass first if you prefer.
WELLS_PER_CELL_TYPE = 48
SITES_PER_WELL = 9
CROPS_PER_SITE = 30

manifest = build_manifest(CACHE_DIR)
subset = subset_manifest(manifest, wells_per_cell_type=WELLS_PER_CELL_TYPE,
                         sites_per_well=SITES_PER_WELL, seed=0)
train_manifest, val_manifest = well_level_split(subset, val_frac=0.2, seed=0)
print(f'train: {len(train_manifest)} sites, val: {len(val_manifest)} sites')
print(f'~{len(train_manifest) * CROPS_PER_SITE:,} train crops (upper bound, before min-cell filtering)')

In [ ]:
# Parallel-prefetch every referenced TIFF + Cells.csv into CACHE_DIR so the GPU
# is not starved on S3 during training. Reuses any cache from a prior session
# (existing files are skipped) and TOLERATES objects missing in the bucket: some
# sites have no Cells.csv, and the dataset skips those at train time too, so a
# 404 must not kill the prefetch.
import logging
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError
from concurrent.futures import ThreadPoolExecutor, as_completed

logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)  # silence pool-full noise

PREFETCH_WORKERS = 32
s3_client = boto3.client('s3', config=Config(
    signature_version=UNSIGNED, max_pool_connections=PREFETCH_WORKERS * 2))

def _expected_keys(mf):
    keys = set()
    for _, row in mf.iterrows():
        for stain in CHANNEL_ORDER:
            keys.add(str(row[f'URL_{stain}']).replace(f's3://{BUCKET}/', '').lstrip('/'))
        keys.add(
            f"{WORKSPACE_PREFIX}analysis/{row['batch']}/{row['Metadata_Plate']}/"
            f"analysis/{row['Metadata_Plate']}-{row['Metadata_Well']}-{row['Metadata_Site']}/Cells.csv"
        )
    return keys

def _download_one(key):
    local = CACHE_DIR / key
    if local.exists():
        return 'cached'
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        s3_client.download_file(BUCKET, key, str(local))
        return 'downloaded'
    except ClientError:
        # Missing in the bucket (e.g. a site with no Cells.csv). Skip it; the
        # dataset skips such sites at train time too.
        if local.exists():
            local.unlink()
        return 'missing'

keys = sorted(_expected_keys(train_manifest) | _expected_keys(val_manifest))
counts = {'cached': 0, 'downloaded': 0, 'missing': 0}
with ThreadPoolExecutor(max_workers=PREFETCH_WORKERS) as ex:
    futures = [ex.submit(_download_one, k) for k in keys]
    for i, fut in enumerate(as_completed(futures)):
        counts[fut.result()] += 1
        if (i + 1) % 2000 == 0:
            print(f"{i + 1}/{len(keys)} processed  {counts}")
print(f"prefetch done: {len(keys)} keys  {counts}")

In [ ]:
from torch.utils.data import DataLoader

torch.set_float32_matmul_precision('high')
torch.manual_seed(0)

BATCH_SIZE = 64
NUM_WORKERS = 8

train_ds = NeuroPaintingDataset(
    train_manifest, CACHE_DIR, crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=3, augment=True,
)
val_ds = NeuroPaintingDataset(
    val_manifest, CACHE_DIR, crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=3, augment=False, shuffle=False,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          persistent_workers=True, pin_memory=True, prefetch_factor=4)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                        persistent_workers=True, pin_memory=True, prefetch_factor=4)

# IterableDataset: train() needs steps_per_epoch. Upper bound from the manifest;
# the loop ends early if min-cell filtering yields fewer crops, which is fine.
steps_per_epoch = max(1, (len(train_manifest) * CROPS_PER_SITE) // BATCH_SIZE)
print(f'steps_per_epoch (upper bound): {steps_per_epoch}')


import json
def report_best_epoch(name, ckpt_dir):
    """Read the per-epoch val records from train_log.jsonl and print the best."""
    best = None
    with open(Path(ckpt_dir) / 'train_log.jsonl') as f:
        for line in f:
            r = json.loads(line)
            if r.get('split') == 'val' and (best is None or r['acc_line_condition'] > best['acc_line_condition']):
                best = r
    if best is None:
        print(f'{name}: no val records found'); return None
    print(f'{name} best-epoch val_acc_line_condition: {best["acc_line_condition"]:.4f} '
          f'(epoch {best["epoch"]}, val_loss {best["L_line_condition"]:.4f})')
    return best

## § A — Argus-RN34 (ResNet34, ImageNet-pretrained)

Run on **Colab Pro (L4)**. Discriminative LR (encoder at 0.1x the head LR) protects the ImageNet features; ~15 epochs with 5% warmup + cosine, matching the recipe that took the Phase 1 baseline to val_acc 0.73.

In [ ]:
from argus_cells.model import BaselineDiseaseClassifier
from argus_cells.training import TrainConfig, train

rn34 = BaselineDiseaseClassifier(in_channels=6, n_classes=2, pretrained_encoder=True)
print(f'Argus-RN34 params: {rn34.parameter_count()}')

rn34_cfg = TrainConfig(
    n_epochs=15,
    steps_per_epoch=steps_per_epoch,
    lr=3e-4,
    encoder_lr_ratio=0.1,     # pretrained encoder gets 0.1x the head LR
    weight_decay=1e-4,
    warmup_steps=max(1, (15 * steps_per_epoch) // 20),  # ~5% warmup
    grad_clip_norm=1.0,
    amp=True,
    seed=0,
)
rn34_dir = CKPT_BASE / 'argus-rn34-v0'
summary_rn34 = train(
    rn34, train_loader, rn34_cfg, checkpoint_dir=rn34_dir,
    val_loader=val_loader, hf_repo='patrickjreed/argus-rn34-v0', device=device,
)
print(summary_rn34)
report_best_epoch('Argus-RN34', rn34_dir)

## § B — Argus-CCT (Compact Convolutional Transformer, from scratch)

Run on a **rented A100** (~1-2 hr). From scratch (no pretraining), so uniform LR and more epochs than RN34, with higher weight decay (transformer regularization) and 10% warmup. The 16x conv tokenizer keeps the 256x256 crops at a 256-token sequence.

In [ ]:
from argus_cells.models.cct import ArgusCCT
from argus_cells.training import TrainConfig, train

cct = ArgusCCT(in_channels=6, n_classes=2, img_size=256)  # 16x16 = 256 tokens
print(f'Argus-CCT params: {cct.parameter_count()}')

cct_cfg = TrainConfig(
    n_epochs=30,              # from scratch needs more epochs than pretrained RN34
    steps_per_epoch=steps_per_epoch,
    lr=6e-4,
    encoder_lr_ratio=1.0,     # uniform LR (CCT has no pretrained encoder to protect)
    weight_decay=5e-2,        # transformers want stronger weight decay
    warmup_steps=max(1, (30 * steps_per_epoch) // 10),  # ~10% warmup
    grad_clip_norm=1.0,
    amp=True,
    seed=0,
)
cct_dir = CKPT_BASE / 'argus-cct-v0'
summary_cct = train(
    cct, train_loader, cct_cfg, checkpoint_dir=cct_dir,
    val_loader=val_loader, hf_repo='patrickjreed/argus-cct-v0', device=device,
)
print(summary_cct)
report_best_epoch('Argus-CCT', cct_dir)

## Done

Both models pushed per-epoch checkpoints to Hugging Face (`patrickjreed/argus-rn34-v0`, `patrickjreed/argus-cct-v0`). **Success gate (spec §6):** at least one of the two reaches val_acc >= 0.73. If both materially underperform, treat it as a training problem (pause for scope review), not an interpretability problem.

Next: `notebooks/06_phase_3_analysis.ipynb` runs the full interpretability harness (channel ablation, GradCAM on RN34, attention rollout on CCT, IG on both, the now-real donor probe, cell-type stratification, cross-architecture agreement) on these two checkpoints.